# Multi-Origin GRU Trajectory State Encoder

This notebook implements a self-supervised GRU-based trajectory state encoder for multi-horizon future-state contrastive retrieval, progress prediction, and optional masked reconstruction, following the advanced logic described in the project prompt.

**Baseline structure and evaluation routines are preserved for direct comparison.**

- W&B project: `gru-multi-origin-2`
- W&B entity: `proj_vla`


In [1]:
# ── Imports, Device, and Seed ───────────────────────────────────────────────
import json
import math
import os
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
import wandb

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Random seed: {SEED}")

Using device: cuda
Random seed: 42


In [2]:
# ── Configuration ───────────────────────────────────────────────────────────
config = {
    # Data
    'data_path': 'train_r2r_img_motion_cleaned_sorted.json',
    'split_ratio_train': 0.8,
    'split_ratio_val': 0.1,
    'split_seed': 42,
    # Motion encoding
    'step_m': 0.25,
    'turn_deg': 15.0,
    # Model
    'input_dim': 3,
    'hidden_dim': 128,
    'embedding_dim': 64,
    'num_layers': 1,
    'dropout': 0.0,
    # Training
    'num_epochs': 100,
    'eval_every': 5,
    'batch_size': 64,
    'min_prefix_len': 2,
    'future_horizons': (1, 5, 10),
    'max_prefixes_per_traj': 3,
    'w_min': 0.3,
    'temperature': 0.07,
    'grad_clip_max_norm': 1.0,
    # Loss weights
    'alpha': 1.0,
    'beta': 0.2,
    'gamma': 0.1,
    # Optimization
    'lr': 1e-3,
    # Logging
    'wandb_entity': 'proj_vla',
    'wandb_project': 'gru-multi-origin-2',
    'wandb_run_name': 'multi-origin-gru',
    'checkpoint_dir': './checkpoints',
    'best_model_path': './checkpoints/best_model.pt',
    'final_model_path': './checkpoints/final_model.pt',
}

print("Config:")
for k, v in config.items():
    print(f"  {k}: {v}")

Config:
  data_path: train_r2r_img_motion_cleaned_sorted.json
  split_ratio_train: 0.8
  split_ratio_val: 0.1
  split_seed: 42
  step_m: 0.25
  turn_deg: 15.0
  input_dim: 3
  hidden_dim: 128
  embedding_dim: 64
  num_layers: 1
  dropout: 0.0
  num_epochs: 100
  eval_every: 5
  batch_size: 64
  min_prefix_len: 2
  future_horizons: (1, 5, 10)
  max_prefixes_per_traj: 3
  w_min: 0.3
  temperature: 0.07
  grad_clip_max_norm: 1.0
  alpha: 1.0
  beta: 0.2
  gamma: 0.1
  lr: 0.001
  wandb_entity: proj_vla
  wandb_project: gru-multi-origin-2
  wandb_run_name: multi-origin-gru
  checkpoint_dir: ./checkpoints
  best_model_path: ./checkpoints/best_model.pt
  final_model_path: ./checkpoints/final_model.pt


In [3]:
# ── Data Loading and Raw Inspection ─────────────────────────────────────────
with open(config['data_path'], 'r') as f:
    raw_data = json.load(f)

print(f"Raw sequences loaded: {len(raw_data)}")

# Extract motion sequences and video_ids
motions = []
video_ids = []
for item in raw_data:
    motions.append(item['motion'])
    video_ids.append(item.get('video_id', None))

lengths = [len(m) for m in motions]
print(f"Sample trajectory length: {lengths[0] if lengths else 'N/A'}")
print(f"Length: min={min(lengths)}, max={max(lengths)}, mean={np.mean(lengths):.1f}")
print(f"Sample motion sequence: {motions[0] if motions else 'N/A'}")

Raw sequences loaded: 10819
Sample trajectory length: 39
Length: min=20, max=501, mean=55.6
Sample motion sequence: [0, 3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 0]


In [4]:
# ── Sequence Cleaning ────────────────────────────────────────────────────────
def clean_motion(seq):
    seq = list(seq)
    # Remove leading STOP tokens (0)
    while seq and seq[0] == 0:
        seq = seq[1:]
    return seq

seen = set()
cleaned_motions = []
for m in motions:
    cleaned = clean_motion(m)
    key = tuple(cleaned)
    if key not in seen and len(cleaned) >= config['min_prefix_len'] + max(config['future_horizons']):
        seen.add(key)
        cleaned_motions.append(cleaned)

cleaned_lengths = [len(m) for m in cleaned_motions]
print(f"Cleaned trajectories: {len(cleaned_motions)}")
print(f"Length: min={min(cleaned_lengths)}, max={max(cleaned_lengths)}, mean={np.mean(cleaned_lengths):.1f}")

Cleaned trajectories: 3600
Length: min=19, max=500, mean=54.6


In [5]:
# ── Action-to-Feature Conversion ─────────────────────────────────────────────
STOP = 0
FORWARD = 1
TURN_LEFT = 2
TURN_RIGHT = 3

step_m = config['step_m']
turn_rad = math.radians(config['turn_deg'])

def actions_to_motion_features(action_seq, theta0=0.0):
    """
    Convert discrete action sequence into motion features [dx, sin(dtheta), cos(dtheta)].
    """
    theta = theta0
    feat_rows = []
    for a in action_seq:
        dx = 0.0
        dtheta = 0.0
        if a == FORWARD:
            dx = step_m
        elif a == TURN_LEFT:
            dtheta = turn_rad
        elif a == TURN_RIGHT:
            dtheta = -turn_rad
        elif a == STOP:
            pass
        else:
            raise ValueError(f"Unknown action id: {a}")
        theta += dtheta
        feat_rows.append([dx, math.sin(dtheta), math.cos(dtheta)])
    return torch.tensor(feat_rows, dtype=torch.float32)

motion_feature_tensors = [actions_to_motion_features(seq) for seq in cleaned_motions]
raw_lengths = torch.tensor([t.size(0) for t in motion_feature_tensors], dtype=torch.long)

print(f"Converted to motion features: {len(motion_feature_tensors)}")
print(f"First feature tensor shape: {motion_feature_tensors[0].shape}")

Converted to motion features: 3600
First feature tensor shape: torch.Size([38, 3])


In [6]:
# ── Train / Validation / Test Split ──────────────────────────────────────────
def split_trajectories(sequences, lengths, train_ratio=0.8, val_ratio=0.1, seed=42):
    idxs = list(range(len(sequences)))
    random.Random(seed).shuffle(idxs)
    n = len(sequences)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train_idx = idxs[:n_train]
    val_idx = idxs[n_train:n_train+n_val]
    test_idx = idxs[n_train+n_val:]
    train = [sequences[i] for i in train_idx]
    val = [sequences[i] for i in val_idx]
    test = [sequences[i] for i in test_idx]
    train_lengths = [lengths[i] for i in train_idx]
    val_lengths = [lengths[i] for i in val_idx]
    test_lengths = [lengths[i] for i in test_idx]
    return (train, train_lengths, train_idx), (val, val_lengths, val_idx), (test, test_lengths, test_idx)

(train_seqs, train_lengths, train_idx), (val_seqs, val_lengths, val_idx), (test_seqs, test_lengths, test_idx) = split_trajectories(
    motion_feature_tensors, raw_lengths.tolist(),
    train_ratio=config['split_ratio_train'],
    val_ratio=config['split_ratio_val'],
    seed=config['split_seed']
)

print(f"Train: {len(train_seqs)}, Val: {len(val_seqs)}, Test: {len(test_seqs)}")
print(f"Train avg len: {np.mean([t.size(0) for t in train_seqs]):.1f}")
print(f"Val avg len: {np.mean([t.size(0) for t in val_seqs]):.1f}")
print(f"Test avg len: {np.mean([t.size(0) for t in test_seqs]):.1f}")

Train: 2880, Val: 360, Test: 360
Train avg len: 54.5
Val avg len: 55.6
Test avg len: 54.3


In [7]:
# ── Prefix / Future Query Specification ─────────────────────────────────────--
def sample_valid_prefix_endpoints(seq_len, min_prefix_len, max_horizon, max_prefixes):
    valid = []
    for t in range(min_prefix_len, seq_len - max_horizon):
        valid.append(t)
    if len(valid) > max_prefixes:
        valid = random.sample(valid, max_prefixes)
    return valid

def build_future_segments(seq, t, horizons):
    futures = []
    for h in horizons:
        if t + h < len(seq):
            futures.append(seq[t:t+h])
        else:
            futures.append(None)
    return futures

# Example usage for one trajectory
example_seq = train_seqs[0]
endpoints = sample_valid_prefix_endpoints(len(example_seq), config['min_prefix_len'], max(config['future_horizons']), config['max_prefixes_per_traj'])
print(f"Sampled prefix endpoints: {endpoints}")
for t in endpoints:
    print(f"Prefix (len={t}):", example_seq[:t].shape)
    print(f"Futures:", [f.shape if f is not None else None for f in build_future_segments(example_seq, t, config['future_horizons'])])

Sampled prefix endpoints: [9, 3, 19]
Prefix (len=9): torch.Size([9, 3])
Futures: [torch.Size([1, 3]), torch.Size([5, 3]), torch.Size([10, 3])]
Prefix (len=3): torch.Size([3, 3])
Futures: [torch.Size([1, 3]), torch.Size([5, 3]), torch.Size([10, 3])]
Prefix (len=19): torch.Size([19, 3])
Futures: [torch.Size([1, 3]), torch.Size([5, 3]), torch.Size([10, 3])]


In [8]:
# ── Multi-Prefix Sampler ─────────────────────────────────────────────────────
def sample_prefix_future_pairs(seqs, horizons, min_prefix_len, max_prefixes, w_min=0.3, return_metadata=True):
    prefixes, futures, prefix_lens, future_lens, progress_targets, traj_ids, timestep_ids, weights = [], [], [], [], [], [], [], []
    for traj_id, seq in enumerate(seqs):
        valid_ts = sample_valid_prefix_endpoints(len(seq), min_prefix_len, max(horizons), max_prefixes)
        for t in valid_ts:
            prefix = seq[:t]
            future_chunks = build_future_segments(seq, t, horizons)
            if any(f is None or len(f) == 0 for f in future_chunks):
                continue
            prefixes.append(prefix)
            futures.append(future_chunks)
            prefix_lens.append(len(prefix))
            future_lens.append([len(f) for f in future_chunks])
            progress = t / (len(seq) - 1)
            progress_targets.append(progress)
            traj_ids.append(traj_id)
            timestep_ids.append(t)
            weight = w_min + (1 - w_min) * progress
            weights.append(weight)
    if not return_metadata:
        return prefixes, futures, prefix_lens, future_lens, progress_targets, weights
    return prefixes, futures, prefix_lens, future_lens, progress_targets, traj_ids, timestep_ids, weights

# Example batch
prefixes, futures, prefix_lens, future_lens, progress_targets, traj_ids, timestep_ids, weights = sample_prefix_future_pairs(
    train_seqs, config['future_horizons'], config['min_prefix_len'], config['max_prefixes_per_traj'], config['w_min'])
print(f"Sampled {len(prefixes)} prefix-future pairs from train set")

Sampled 8640 prefix-future pairs from train set


In [9]:
# ── Dataset / Batching Utilities ─────────────────────────────────────────────
class TrajPrefixFutureDataset(Dataset):
    def __init__(self, seqs, horizons, min_prefix_len, max_prefixes, w_min=0.3):
        self.prefixes, self.futures, self.prefix_lens, self.future_lens, self.progress_targets, self.traj_ids, self.timestep_ids, self.weights = sample_prefix_future_pairs(
            seqs, horizons, min_prefix_len, max_prefixes, w_min=w_min, return_metadata=True)

    def __len__(self):
        return len(self.prefixes)

    def __getitem__(self, idx):
        return {
            'prefix': self.prefixes[idx],
            'futures': self.futures[idx],
            'prefix_len': self.prefix_lens[idx],
            'future_lens': self.future_lens[idx],
            'progress': self.progress_targets[idx],
            'traj_id': self.traj_ids[idx],
            'timestep': self.timestep_ids[idx],
            'weight': self.weights[idx],
        }

def collate_fn(batch):
    prefix_seqs = [x['prefix'].detach().clone() if torch.is_tensor(x['prefix']) else torch.tensor(x['prefix'], dtype=torch.float32) for x in batch]
    prefix_lens = torch.tensor([x['prefix_len'] for x in batch], dtype=torch.long)
    progress = torch.tensor([x['progress'] for x in batch], dtype=torch.float32)
    weights = torch.tensor([x['weight'] for x in batch], dtype=torch.float32)
    traj_ids = torch.tensor([x['traj_id'] for x in batch], dtype=torch.long)
    timestep_ids = torch.tensor([x['timestep'] for x in batch], dtype=torch.long)
    # Futures: list of [num_horizons] lists
    num_horizons = len(batch[0]['futures'])
    future_seqs = [[x['futures'][h].detach().clone() if torch.is_tensor(x['futures'][h]) else torch.tensor(x['futures'][h], dtype=torch.float32) for x in batch] for h in range(num_horizons)]
    future_lens = [[len(x['futures'][h]) for x in batch] for h in range(num_horizons)]
    # Pad
    prefix_padded = pad_sequence(prefix_seqs, batch_first=True)
    future_padded = [pad_sequence(future_seqs[h], batch_first=True) for h in range(num_horizons)]
    return {
        'prefix': prefix_padded,
        'prefix_len': prefix_lens,
        'futures': future_padded,
        'future_lens': future_lens,
        'progress': progress,
        'weights': weights,
        'traj_ids': traj_ids,
        'timestep_ids': timestep_ids,
    }

# Example DataLoader
train_dataset = TrajPrefixFutureDataset(train_seqs, config['future_horizons'], config['min_prefix_len'], config['max_prefixes_per_traj'], config['w_min'])
train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, collate_fn=collate_fn)
batch = next(iter(train_loader))
print(f"Batch prefix shape: {batch['prefix'].shape}")
print(f"Batch future[0] shape: {batch['futures'][0].shape}")

Batch prefix shape: torch.Size([64, 72, 3])
Batch future[0] shape: torch.Size([64, 1, 3])


In [10]:
# ── GRU Prefix Encoder ─────────────────────────────────────────────────────--
class GRUPrefixEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, embedding_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim)
        )
    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        h_last = h_n[-1]  # [B, hidden_dim]
        z = self.proj(h_last)
        z = F.normalize(z, dim=-1)
        return z, h_last

# Example
prefix_encoder = GRUPrefixEncoder(
    input_dim=config['input_dim'],
    hidden_dim=config['hidden_dim'],
    embedding_dim=config['embedding_dim'],
    num_layers=config['num_layers'],
    dropout=config['dropout']
).to(device)

z, h = prefix_encoder(batch['prefix'].to(device), batch['prefix_len'].to(device))
print(f"Prefix encoder output shape: {z.shape}")

Prefix encoder output shape: torch.Size([64, 64])


In [11]:
# ── GRU Future Encoder ─────────────────────────────────────────────────────--
class GRUFutureEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, embedding_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim)
        )
    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        h_last = h_n[-1]  # [B, hidden_dim]
        z = self.proj(h_last)
        z = F.normalize(z, dim=-1)
        return z, h_last

# Note: Separate future encoder prevents shortcut sharing and enforces asymmetric representation learning.

# Example
future_encoder = GRUFutureEncoder(
    input_dim=config['input_dim'],
    hidden_dim=config['hidden_dim'],
    embedding_dim=config['embedding_dim'],
    num_layers=config['num_layers'],
    dropout=config['dropout']
).to(device)

z_f, h_f = future_encoder(batch['futures'][0].to(device), torch.tensor(batch['future_lens'][0]).to(device))
print(f"Future encoder output shape: {z_f.shape}")

Future encoder output shape: torch.Size([64, 64])


In [12]:
# ── Progress Head ─────────────────────────────────────────────────────────---
class ProgressHead(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, h):
        return self.fc(h)

# Example
progress_head = ProgressHead(config['hidden_dim']).to(device)
pred_progress = progress_head(h)
print(f"Progress prediction shape: {pred_progress.shape}")

Progress prediction shape: torch.Size([64, 1])


In [13]:
# ── Optional Masked Prefix Reconstruction Head ─────────────────────────────---
class MaskedPrefixReconstructionHead(nn.Module):
    def __init__(self, hidden_dim, input_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
    def forward(self, h):
        return self.fc(h)

def mask_prefix_chunk(prefix, mask_ratio=0.15):
    T = prefix.size(1)
    mask_len = max(1, int(T * mask_ratio))
    start = random.randint(0, T - mask_len)
    mask = torch.zeros(T, dtype=torch.bool)
    mask[start:start+mask_len] = True
    return mask

# Example (optional, only if using mask loss)
# mask_head = MaskedPrefixReconstructionHead(config['hidden_dim'], config['input_dim']).to(device)


In [14]:
# ── Full Model Wrapper ─────────────────────────────────────────────────────---
class TrajectoryStateModel(nn.Module):
    def __init__(self, config, use_mask_loss=False):
        super().__init__()
        self.prefix_encoder = GRUPrefixEncoder(
            config['input_dim'], config['hidden_dim'], config['embedding_dim'], config['num_layers'], config['dropout'])
        self.future_encoder = GRUFutureEncoder(
            config['input_dim'], config['hidden_dim'], config['embedding_dim'], config['num_layers'], config['dropout'])
        self.progress_head = ProgressHead(config['hidden_dim'])
        self.use_mask_loss = use_mask_loss
        if use_mask_loss:
            self.mask_head = MaskedPrefixReconstructionHead(config['hidden_dim'], config['input_dim'])
    def encode_prefix(self, x, lengths):
        z, h = self.prefix_encoder(x, lengths)
        return z, h
    def encode_future(self, x, lengths):
        z, h = self.future_encoder(x, lengths)
        return z, h
    def predict_progress(self, h):
        return self.progress_head(h)
    def reconstruct_masked(self, h):
        if self.use_mask_loss:
            return self.mask_head(h)
        else:
            return None

# Example
model = TrajectoryStateModel(config, use_mask_loss=False).to(device)


In [15]:
# ── Contrastive Retrieval Loss (InfoNCE) ─────────────────────────────────-----
def infonce_loss(z_prefix, z_future, temperature=0.07, sample_weights=None, traj_ids=None, timestep_ids=None, same_traj_mask=True):
    # z_prefix, z_future: [B, D] normalized
    logits = torch.matmul(z_prefix, z_future.T) / temperature
    targets = torch.arange(z_prefix.size(0), device=z_prefix.device)
    mask = torch.ones_like(logits, dtype=torch.bool)
    if same_traj_mask and traj_ids is not None:
        for i in range(len(traj_ids)):
            for j in range(len(traj_ids)):
                if i != j and traj_ids[i] == traj_ids[j]:
                    mask[i, j] = False
    logits_masked = logits.masked_fill(~mask, float('-inf'))
    loss = F.cross_entropy(logits_masked, targets, reduction='none')
    if sample_weights is not None:
        loss = loss * sample_weights
    return loss.mean()


In [16]:
# ── Negative Sampling Utilities ─────────────────────────────────────────-----
def build_negative_masks(traj_ids, future_start_ids, hard_window=2):
    # Returns a mask [B, B] where False means invalid negative
    B = len(traj_ids)
    mask = torch.ones((B, B), dtype=torch.bool)
    for i in range(B):
        for j in range(B):
            if i == j:
                continue
            if traj_ids[i] == traj_ids[j]:
                # Same trajectory
                if abs(future_start_ids[i] - future_start_ids[j]) <= hard_window:
                    mask[i, j] = False  # Too close, not a hard negative
    return mask


In [17]:
# ── Auxiliary Losses ─────────────────────────────────────────────────--------
def progress_loss(pred_progress, target_progress):
    return F.mse_loss(pred_progress.squeeze(-1), target_progress)

def mask_loss_fn(recon, target, mask):
    # recon, target: [B, T, D], mask: [B, T]
    loss = ((recon - target) ** 2).sum(dim=-1)
    loss = (loss * mask.float()).sum() / (mask.float().sum() + 1e-8)
    return loss


In [18]:
# ── Retrieval Metrics ─────────────────────────────────────────────────-------
def compute_ranks(logits, positive_indices):
    # logits: [B, N], positive_indices: [B]
    sorted_idx = logits.argsort(dim=-1, descending=True)
    ranks = [(sorted_idx[i] == positive_indices[i]).nonzero(as_tuple=True)[0].item() + 1 for i in range(len(positive_indices))]
    return ranks

def retrieval_metrics_from_logits(logits, positive_indices, ks=(1, 5, 10)):
    ranks = compute_ranks(logits, positive_indices)
    metrics = {}
    for k in ks:
        metrics[f'top{k}'] = np.mean([r <= k for r in ranks])
    metrics['mrr'] = np.mean([1.0 / r for r in ranks])
    return metrics


In [19]:
# ── Progress Metrics ─────────────────────────────────────────────────--------
def compute_progress_metrics(pred, target):
    pred = pred.squeeze(-1).detach().cpu().numpy()
    target = target.detach().cpu().numpy()
    mae = np.mean(np.abs(pred - target))
    rmse = np.sqrt(np.mean((pred - target) ** 2))
    return {'mae': mae, 'rmse': rmse}


In [20]:
# ── Training Epoch Function ─────────────────────────────────────────────────-
def train_epoch(model, loader, optimizer, config, device, use_mask_loss=False):
    model.train()
    total_loss, future_loss, prog_loss, mask_loss_val = 0, 0, 0, 0
    all_top1, all_top5, all_mrr, all_mae = [], [], [], []
    for batch in loader:
        optimizer.zero_grad()
        prefix = batch['prefix'].to(device)
        prefix_len = batch['prefix_len'].to(device)
        progress = batch['progress'].to(device)
        weights = batch['weights'].to(device)
        traj_ids = batch['traj_ids'].to(device)
        timestep_ids = batch['timestep_ids'].to(device)
        # Encode prefix
        z_p, h_p = model.encode_prefix(prefix, prefix_len)
        # Encode future (use first horizon for training)
        future = batch['futures'][0].to(device)
        future_len = torch.tensor(batch['future_lens'][0]).to(device)
        z_f, _ = model.encode_future(future, future_len)
        # Retrieval loss
        loss_retrieval = infonce_loss(z_p, z_f, config['temperature'], weights, traj_ids, timestep_ids)
        # Progress loss
        pred_progress = model.predict_progress(h_p)
        loss_prog = progress_loss(pred_progress, progress)
        # Optional mask loss
        if use_mask_loss:
            mask = mask_prefix_chunk(prefix)
            recon = model.reconstruct_masked(h_p)
            loss_mask = mask_loss_fn(recon, prefix, mask)
        else:
            loss_mask = 0.0
        # Total loss
        total = config['alpha'] * loss_retrieval + config['beta'] * loss_prog + (config['gamma'] * loss_mask if use_mask_loss else 0.0)
        total.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip_max_norm'])
        optimizer.step()
        # Metrics
        total_loss += total.item()
        future_loss += loss_retrieval.item()
        prog_loss += loss_prog.item()
        mask_loss_val += loss_mask if use_mask_loss else 0.0
        # Retrieval metrics
        logits = torch.matmul(z_p, z_f.T)
        metrics = retrieval_metrics_from_logits(logits, torch.arange(z_p.size(0)))
        all_top1.append(metrics['top1'])
        all_top5.append(metrics['top5'])
        all_mrr.append(metrics['mrr'])
        # Progress metrics
        prog_metrics = compute_progress_metrics(pred_progress, progress)
        all_mae.append(prog_metrics['mae'])
    n = len(loader)
    return {
        'loss': total_loss / n,
        'future_loss': future_loss / n,
        'prog_loss': prog_loss / n,
        'mask_loss': mask_loss_val / n if use_mask_loss else 0.0,
        'top1': np.mean(all_top1),
        'top5': np.mean(all_top5),
        'mrr': np.mean(all_mrr),
        'prog_mae': np.mean(all_mae),
    }


In [21]:
# ── W&B Setup ─────────────────────────────────────────────────────────------
wandb.init(
    project=config['wandb_project'],
    entity=config['wandb_entity'],
    name=config['wandb_run_name'],
    config=config,
    )
print(f"W&B run URL: {wandb.run.get_url()}")

# Log a dummy metric to ensure W&B visualizations appear
wandb.log({"dummy_metric": 0}, step=0)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/aakash/.netrc.
wandb: Currently logged in as: agurram1 (agurram1-johns-hopkins-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.


W&B run URL: https://wandb.ai/proj_vla/gru-multi-origin-2/runs/6a7b8ulk


In [22]:
# ── Full Training Loop ─────────────────────────────────────────────────------
model = TrajectoryStateModel(config, use_mask_loss=False).to(device)
optimizer = optim.Adam(model.parameters(), lr=config['lr'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['num_epochs'])
best_top1 = 0.0
for epoch in range(1, config['num_epochs'] + 1):
    train_metrics = train_epoch(model, train_loader, optimizer, config, device, use_mask_loss=False)
    scheduler.step()
    wandb.log({f"train/{k}": v for k, v in train_metrics.items()}, step=epoch)
    print(f"epoch {epoch:03d} | train_loss={train_metrics['loss']:.4f} | future_loss={train_metrics['future_loss']:.4f} | prog_loss={train_metrics['prog_loss']:.4f} | top1={train_metrics['top1']:.3f} | top5={train_metrics['top5']:.3f} | mrr={train_metrics['mrr']:.3f} | prog_mae={train_metrics['prog_mae']:.3f}")
    # Save best model
    if train_metrics['top1'] > best_top1:
        best_top1 = train_metrics['top1']
        torch.save(model.state_dict(), config['best_model_path'])
        print(f"Saved new best model at epoch {epoch}")
    # Optionally save final model
    if epoch == config['num_epochs']:
        torch.save(model.state_dict(), config['final_model_path'])
        print(f"Saved final model at epoch {epoch}")


epoch 001 | train_loss=2.4141 | future_loss=2.4049 | prog_loss=0.0458 | top1=0.027 | top5=0.135 | mrr=0.104 | prog_mae=0.180
Saved new best model at epoch 1
epoch 002 | train_loss=2.3911 | future_loss=2.3874 | prog_loss=0.0183 | top1=0.026 | top5=0.145 | mrr=0.107 | prog_mae=0.107
epoch 003 | train_loss=2.3890 | future_loss=2.3862 | prog_loss=0.0141 | top1=0.028 | top5=0.141 | mrr=0.108 | prog_mae=0.092
Saved new best model at epoch 3
epoch 004 | train_loss=2.3865 | future_loss=2.3841 | prog_loss=0.0115 | top1=0.031 | top5=0.143 | mrr=0.110 | prog_mae=0.084
Saved new best model at epoch 4
epoch 005 | train_loss=2.3845 | future_loss=2.3820 | prog_loss=0.0128 | top1=0.031 | top5=0.148 | mrr=0.111 | prog_mae=0.089
Saved new best model at epoch 5
epoch 006 | train_loss=2.3833 | future_loss=2.3809 | prog_loss=0.0119 | top1=0.030 | top5=0.142 | mrr=0.109 | prog_mae=0.085
epoch 007 | train_loss=2.3829 | future_loss=2.3804 | prog_loss=0.0123 | top1=0.028 | top5=0.145 | mrr=0.108 | prog_mae=0.0